# Ejercicio 1 — Aprendizaje Supervisado
## Predicción de renuncia de empleados

Notebook narrativo (capa de presentación). **Toda la lógica vive en `pipeline/`**;
aquí solo se importa, se ejecuta y se explica. Equivale a `python main.py` pero
mostrando cada paso celda por celda (para el video).


In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))  # hacer visible pipeline/ desde notebooks/
%matplotlib inline

from pipeline import config, data_generation, eda, preprocessing, models, evaluation, tuning

## 1. Creación del dataset

Dataset sintético con **lógica de coherencia** (no aleatorio puro): la satisfacción
depende de horas extra y distancia; el salario de edad/antigüedad/evaluación; y la
renuncia de los factores de riesgo combinados.

In [ ]:
df = data_generation.generar_dataset()
print(f'{len(df)} registros')
for r in data_generation.REGLAS_COHERENCIA:
    print(' -', r)
df.head()

## 2. Análisis exploratorio (EDA)

In [ ]:
eda.estadisticas_descriptivas(df).round(2)

In [ ]:
print('Top 3 correlaciones con renuncia:')
print(eda.correlaciones_top(df).round(3))
print('\nBalance de clases:', eda.balance_clases(df).round(3).to_dict())

In [ ]:
_ = eda.fig_heatmap_correlacion(df)

In [ ]:
_ = eda.fig_distribucion_target(df)

In [ ]:
_ = eda.fig_boxplots(df)

## 3. Modelado — 3 algoritmos

Split 80/20 (`random_state=42`); `StandardScaler` ajustado **solo** en train.

In [ ]:
datos = preprocessing.split_y_escalar(df)
modelos = models.entrenar_modelos(datos.X_train, datos.y_train)
evaluation.tabla_metricas(modelos, datos.X_test, datos.y_test)

In [ ]:
# Curva ROC de los 3 modelos (pregunta de video: por qué un modelo domina a otro)
_ = evaluation.fig_roc_comparativa(modelos, datos.X_test, datos.y_test)

In [ ]:
tabla = evaluation.tabla_metricas(modelos, datos.X_test, datos.y_test)
mejor = tuning.seleccionar_mejor(tabla)
print('Mejor modelo:', mejor)
_ = evaluation.fig_matriz_confusion(modelos[mejor], datos.X_test, datos.y_test, mejor)

In [ ]:
# Feature importance (Random Forest / Gradient Boosting)
_ = evaluation.fig_feature_importance(modelos['Random Forest'], datos.feature_names, 'Random Forest')

## 4. Optimización con Cross-Validation

In [ ]:
cv_mean, cv_std = tuning.cross_validation_f1(modelos[mejor], datos.X_train, datos.y_train)
print(f'CV F1 (k={config.CV_FOLDS}) = {cv_mean:.4f} +/- {cv_std:.4f}')

busqueda = tuning.optimizar(mejor, modelos[mejor], datos.X_train, datos.y_train)
print('Mejores hiperparámetros:', busqueda.best_params_)
tuning.comparar_base_vs_opt(modelos[mejor], busqueda.best_estimator_, datos.X_test, datos.y_test)